In [1]:
%cd ../../../

/home/hoanghu/projects/Food-Waste-Optimization


In [2]:
import re
from pathlib import Path
from itertools import zip_longest, product

import numpy as np
import pandas as pd


# Load dim `meal_names`

In [3]:
dim_meal_names = pd.read_excel("data/processed/phase_4/dim_meal_names.xlsx")

dim_meal_names.head()

,meal_id,meal
0,9017,"""Butter"" härkäpapua & pähkinää"
1,7201,2023 Härkäpu-sienilasagnette
2,9032,Appelisiini-luomukikhernecurrya
3,9102,Artisokkavugetteja & tuoretomaattisalsaa
4,7010,Aurajuusto-pinaattilasagnette


In [4]:
co2 = dim_meal_names.copy()

# Get CO2 info from meal list

## Process meals

In [5]:
meals_raw = pd.read_excel("data/processed/phase_4/menus.xlsx", sheet_name="meals")

meals_raw.head()

,meal_code,meal_name,category,CO2
0,34.0,Sitruunaiset kalapaloja ja kukkakaalitsatsikia,kala,0.81
1,710.0,Broileri-Caesarsalaatti,kana,0.67
2,724.0,Broilerilasagnette,kana,0.82
3,725.0,"Broilerinuggetit, currykastiketta",kana,1.06
4,726.0,"Broileripyörykät, currykastike",kana,0.86


In [6]:
# Remove duplicate entries
meal_list = (
    meals_raw
    .drop(columns='category')
    .dropna(axis=0, how='any')
    .groupby('meal_code')
    .last()
    .reset_index()
    .rename(columns={
        'meal_code': 'meal_id',
        'meal_name': 'meal',
        'CO2': 'co2'
    })
)


# Decompose meal name and tag
pat1 = r"(\s?\(\s?[A-Z](\w|\,|\s)*\))"
pat2 = r"(\(|\)|\s)"

def _f_extract_tag(s: str):
    meal_name, tag = "", ""

    s = s.strip()
    out = re.findall(pat1, s)
    if len(out) < 1:
        meal_name = s
    else:
        meal_name = re.sub(pat1, "", s).strip()
        tag = out[0][0]
        tag = re.sub(pat2, '', tag)

    return pd.Series({'meal': meal_name, 'tag': tag})
    
meal_list['meal'] = meal_list['meal'].str.strip().apply(_f_extract_tag)['meal']



meal_list['meal_id'] = meal_list['meal_id'].astype(int)

meal_list = meal_list.sort_values('meal_id').groupby('meal').first().reset_index()

meal_list = meal_list[~meal_list['meal'].str.lower().str.contains('take away')]



meal_list.head()

,meal,meal_id,co2
0,"""Butter"" härkäpapua & pähkinää",9017,0.49
1,2023 Härkäpu-sienilasagnette,7201,0.55
2,Appelisiini-luomukikhernecurrya,9032,0.44
3,Artisokkavugetteja & tuoretomaattisalsaa,9102,0.39
4,Aurajuusto-pinaattilasagnette,7010,0.74


## Merge with meal list

In [7]:
co2 = (
    co2
    .merge(meal_list.rename(columns={'meal': 'meal_1'}), on='meal_id', how='left')
)

co2.head()

,meal_id,meal,meal_1,co2
0,9017,"""Butter"" härkäpapua & pähkinää","""Butter"" härkäpapua & pähkinää",0.49
1,7201,2023 Härkäpu-sienilasagnette,2023 Härkäpu-sienilasagnette,0.55
2,9032,Appelisiini-luomukikhernecurrya,Appelisiini-luomukikhernecurrya,0.44
3,9102,Artisokkavugetteja & tuoretomaattisalsaa,Artisokkavugetteja & tuoretomaattisalsaa,0.39
4,7010,Aurajuusto-pinaattilasagnette,Aurajuusto-pinaattilasagnette,0.74


# Process POS

In [8]:
paths = [
    "data/raw/pos/Sold lunches.csv",
    "data/raw/pos/Sold lunches Kumpula 6-8 2024.csv",
    "data/raw/pos/Sold lunches Kumpula 9-10 2024.csv",
]

raw = []
for path in paths:
    df = pd.read_csv(path, delimiter=';')
    df.columns = np.arange(df.shape[1])
    raw.append(df)


pos_raw = pd.concat(raw, ignore_index=True)
pos_raw.head()

/tmp/ipykernel_52752/1943815052.py:9: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, delimiter=';')


,0,1,2,3,4,5,6
0,2.1.2023,10:31,600 Chemicum,Liha,"Uunimakkaraa,sinappikastiketta",1,"0,9"
1,2.1.2023,10:32,600 Chemicum,Kala,Kalapuikot tillikermaviilikast,1,"1,04"
2,2.1.2023,10:32,600 Chemicum,Liha,"Uunimakkaraa,sinappikastiketta",1,"0,9"
3,2.1.2023,10:35,600 Chemicum,Kala,Kalapuikot tillikermaviilikast,1,"1,04"
4,2.1.2023,10:36,600 Chemicum,Liha,"Uunimakkaraa,sinappikastiketta",2,"1,8"


In [9]:
pos = pos_raw.copy()


# Rename columns
pos.columns = np.arange(pos.shape[1])
pos.rename(
    columns={
        0: 'date',
        1: 'time',
        2: 'restaurant',
        3: 'meal_type',
        4: 'meal',
        5: 'pcs',
        6: 'co2',
    },
    inplace=True
)

# Process meal_type
pos['meal_type'] = pos['meal_type'].map({
    'Liha': 'meat',
    'Kala': 'fish',
    'Vegaani': 'vegan',
    'Kasvis': 'vegetarian',
    'Kana': 'chicken'
})


# Process pcs
pos['pcs'] = pos['pcs'].replace(' ', np.nan).astype(np.float32)

# Process meal
pos['meal'] = pos['meal'].str.strip()


# Process CO2
def _f_process(s: str):
    s = s.replace(',', '.')
    s = s.replace(' ', '')

    out = np.nan
    if s != '':
        out = float(s)

    return out

pos['co2'] = pos['co2'].apply(_f_process)
pos['co2'] /= pos['pcs']


# Filter entries with invalod pcs
pos = pos[
    (~pos['pcs'].isna())
    & (pos['pcs'] > 0)
]

# Get pair of meal and meal_type
meals_pos = pos.groupby(['meal', 'meal_type']).last().reset_index()


# Remove take away
meals_pos = meals_pos[~meals_pos['meal'].str.lower().str.contains('take away')]


# Remove unnecessary columns
meals_pos = meals_pos[['meal', 'co2']]

meals_pos.head()

,meal,co2
0,"""Butter"" luomukikhernekastiketta",0.45
1,Aurajuusto-pinaattilasagnettea,0.74
2,BBQ-Broilerikastiketta,0.57
3,Bangladeshilainen linssipata,0.41
4,Bataatti-maapähkinäkeitto,0.40


## Merge with meals from POS

In [10]:
co2 = (
    co2
    .merge(meals_pos, on='meal', how='left')
)


# Combine 2 CO2 columns
co2['co2'] = co2[['co2_x', 'co2_y']].bfill(axis=1).iloc[:, 0]


# Filter out NaN CO2 rows
co2 = co2[~co2['co2'].isna()]


# Keep necessary columns
co2 = co2[['meal_id', 'co2']]

co2.head()

,meal_id,co2
0,9017,0.49
1,7201,0.55
2,9032,0.44
3,9102,0.39
4,7010,0.74


# Save dim

In [11]:
co2.to_excel("data/processed/phase_4/dim_co2.xlsx", index=False)